<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/Sterimol_and_BuriedVol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# 1. INSTALL
# ============================================================

!pip install -q morfeus-ml numpy scipy pandas openpyxl


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd

from google.colab import drive
from morfeus import Sterimol, BuriedVolume


# ============================================================
# 3. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount('/content/drive')


# ============================================================
# 4. SETTINGS FOR YOUR STRUCTURES
# ============================================================

xyz_folder = "/content/drive/MyDrive/xyz_files"

# Your numbering
C2_ATOM = 18

# Universal first atom of R3
R3_FIRST_ATOM = 44

# Atoms 16–43 are common substrate atoms
ALIGN_START = 16
ALIGN_END   = 43


# ============================================================
# 5. READ XYZ FILE
# ============================================================

def read_xyz(filename):

    with open(filename, "r") as f:
        lines = f.readlines()

    n_atoms = int(lines[0].strip())

    elements = []
    coordinates = []

    for line in lines[2:2+n_atoms]:

        parts = line.split()

        elements.append(parts[0])

        coordinates.append([
            float(parts[1]),
            float(parts[2]),
            float(parts[3])
        ])

    return elements, np.array(coordinates, dtype=float)


# ============================================================
# 6. COVALENT RADII
# ============================================================

covalent_radii = {

    "H":  0.31,
    "C":  0.76,
    "N":  0.71,
    "O":  0.66,
    "F":  0.57,
    "Cl": 1.02,
    "Br": 1.20,
    "I":  1.39,
    "S":  1.05,
    "P":  1.07
}


# ============================================================
# 7. BUILD CONNECTIVITY
# ============================================================

def build_connectivity(elements, coords):

    n = len(elements)

    connected = [[] for _ in range(n)]

    for i in range(n):

        for j in range(i + 1, n):

            ri = covalent_radii.get(elements[i], 0.77)
            rj = covalent_radii.get(elements[j], 0.77)

            distance = np.linalg.norm(
                coords[i] - coords[j]
            )

            cutoff = 1.25 * (ri + rj)

            if distance <= cutoff:

                connected[i].append(j)
                connected[j].append(i)

    return connected


# ============================================================
# 8. FIND R3 FRAGMENT
#
# C2 = atom 18
# R3 first atom = atom 44
#
# We start at C44 and collect all atoms connected to it.
# We specifically prevent the search from entering C18.
# ============================================================

def get_r3_fragment(elements, coords):

    connected = build_connectivity(elements, coords)

    start = R3_FIRST_ATOM - 1
    c2 = C2_ATOM - 1

    fragment = set([start])

    queue = [start]

    while queue:

        current = queue.pop(0)

        for neighbor in connected[current]:

            # NEVER cross back through C2
            if neighbor == c2:
                continue

            if neighbor not in fragment:

                fragment.add(neighbor)

                queue.append(neighbor)

    return sorted(fragment)


# ============================================================
# 9. KABSCH ALIGNMENT
#
# This is only for visualization/superposition.
# It is NOT required for Sterimol calculation.
# ============================================================

def kabsch_transform(P, Q):

    P_center = P.mean(axis=0)
    Q_center = Q.mean(axis=0)

    Pc = P - P_center
    Qc = Q - Q_center

    H = Pc.T @ Qc

    U, S, Vt = np.linalg.svd(H)

    d = np.sign(np.linalg.det(Vt.T @ U.T))

    D = np.eye(3)
    D[2, 2] = d

    R = Vt.T @ D @ U.T

    return R, P_center, Q_center


# ============================================================
# 10. GET XYZ FILES
# ============================================================

xyz_files = sorted([

    f for f in os.listdir(xyz_folder)

    if f.lower().endswith(".xyz")

])


print("====================================================")
print("Number of XYZ files:", len(xyz_files))
print("====================================================")


if len(xyz_files) == 0:

    raise ValueError(
        "No XYZ files found in the specified folder."
    )


# ============================================================
# 11. REFERENCE STRUCTURE
# ============================================================

reference_file = os.path.join(
    xyz_folder,
    xyz_files[0]
)

ref_elements, ref_coords = read_xyz(
    reference_file
)


# ============================================================
# 12. STORAGE
# ============================================================

results = []

aligned_structures = []


# ============================================================
# 13. PROCESS EVERY XYZ FILE
# ============================================================

for filename in xyz_files:

    print("\n----------------------------------------------------")
    print("Processing:", filename)
    print("----------------------------------------------------")

    path = os.path.join(
        xyz_folder,
        filename
    )

    try:

        elements, coords = read_xyz(path)

        n_atoms = len(elements)

        # ----------------------------------------------------
        # CHECK ATOM NUMBERS
        # ----------------------------------------------------

        if n_atoms < R3_FIRST_ATOM:

            print(
                "SKIPPED: fewer than 44 atoms."
            )

            continue


        # ====================================================
        # A. FIND R3
        # ====================================================

        r3_atoms = get_r3_fragment(
            elements,
            coords
        )

        r3_atom_numbers = [
            i + 1 for i in r3_atoms
        ]

        print(
            "R3 atoms:",
            r3_atom_numbers
        )


        # ====================================================
        # B. CHECK C2-R3 BOND
        # ====================================================

        c2_index = C2_ATOM - 1
        r3_first_index = R3_FIRST_ATOM - 1

        c2_r3_distance = np.linalg.norm(
            coords[c2_index] -
            coords[r3_first_index]
        )

        print(
            f"C18-C44 distance = "
            f"{c2_r3_distance:.3f} Å"
        )


        # ====================================================
        # C. STERIMOL
        #
        # Axis:
        #
        # C18  ---------------->  C44
        #
        # ====================================================

        sterimol = Sterimol(

            elements,
            coords,

            # Morfeus uses ZERO-based indices
            c2_index,
            r3_first_index
        )


        B1 = sterimol.B_1_value
        B5 = sterimol.B_5_value
        L  = sterimol.L_value


        print(
            f"Sterimol B1 = {B1:.3f} Å"
        )

        print(
            f"Sterimol B5 = {B5:.3f} Å"
        )

        print(
            f"Sterimol L  = {L:.3f} Å"
        )


        # ====================================================
        # D. BURIED VOLUME
        #
        # We create a system containing:
        #
        # C18 + all R3 atoms
        #
        # Then calculate how much of a 3.5 Å sphere
        # around C18 is occupied by R3.
        #
        # ====================================================

        vbur_indices = [c2_index] + r3_atoms

        vbur_elements = [
            elements[i]
            for i in vbur_indices
        ]

        vbur_coords = np.array([
            coords[i]
            for i in vbur_indices
        ])


        # C18 is the first atom in this new subset
        vbur_center = 1


        buried = BuriedVolume(

            vbur_elements,
            vbur_coords,

            vbur_center,

            # Standard buried-volume sphere
            radius=3.5
        )


        Vbur = buried.buried_volume
        percent_Vbur = buried.fraction_buried_volume * 100


        print(
            f"Vbur = {Vbur:.2f} Å³"
        )

        print(
            f"%Vbur = {percent_Vbur:.2f}%"
        )


        # ====================================================
        # E. STORE RESULTS
        # ====================================================

        results.append({

            "XYZ file": filename,

            "C2 atom": C2_ATOM,

            "R3 first atom": R3_FIRST_ATOM,

            "R3 atom numbers":
                ",".join(map(str, r3_atom_numbers)),

            "C18-C44 distance (Å)":
                c2_r3_distance,

            "Sterimol B1 (Å)": B1,

            "Sterimol B5 (Å)": B5,

            "Sterimol L (Å)": L,

            "Vbur (Å³)": Vbur,

            "%Vbur": percent_Vbur

        })


        # ====================================================
        # F. SAVE ALIGNMENT INFORMATION
        # ====================================================

        if len(ref_coords) >= ALIGN_END:

            P = coords[
                ALIGN_START-1:ALIGN_END
            ]

            Q = ref_coords[
                ALIGN_START-1:ALIGN_END
            ]

            R, P_center, Q_center = \
                kabsch_transform(P, Q)

            aligned_coords = \
                (coords - P_center) @ R + Q_center


            aligned_structures.append({

                "file": filename,

                "elements": elements,

                "coords": aligned_coords,

                "r3_atoms": r3_atoms

            })


    except Exception as e:

        print(
            "ERROR processing",
            filename
        )

        print(e)


# ============================================================
# 14. CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(results)


# ============================================================
# 15. DISPLAY RESULTS
# ============================================================

print("\n\n====================================================")
print("FINAL RESULTS")
print("====================================================")

display(df)


# ============================================================
# 16. SAVE EXCEL
# ============================================================

output_excel = os.path.join(

    xyz_folder,

    "R3_Sterimol_Vbur_results.xlsx"

)


df.to_excel(
    output_excel,
    index=False
)


print("\n====================================================")
print("Excel file saved:")
print(output_excel)
print("====================================================")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of XYZ files: 34

----------------------------------------------------
Processing: RR_CH2-Ph.xyz
----------------------------------------------------
R3 atoms: [44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57]
C18-C44 distance = 1.529 Å
Sterimol B1 = 3.471 Å
Sterimol B5 = 9.484 Å
Sterimol L  = 3.987 Å
Vbur = 48.78 Å³
%Vbur = 27.16%

----------------------------------------------------
Processing: RR_CH2-dioxane.xyz
----------------------------------------------------
R3 atoms: [44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56]
C18-C44 distance = 1.529 Å
Sterimol B1 = 3.367 Å
Sterimol B5 = 9.520 Å
Sterimol L  = 5.213 Å
Vbur = 47.39 Å³
%Vbur = 26.38%

----------------------------------------------------
Processing: RR_CH2CN.xyz
----------------------------------------------------
R3 atoms: [44, 45, 46, 47, 48]
C18-C44 distance = 1.540 Å
Sterimol

,XYZ file,C2 atom,R3 first atom,R3 atom numbers,C18-C44 distance (Å),Sterimol B1 (Å),Sterimol B5 (Å),Sterimol L (Å),Vbur (Å³),%Vbur
0,RR_CH2-Ph.xyz,18,44,"44,45,46,47,48,49,50,51,52,53,54,55,56,57",1.529416,3.470654,9.483736,3.986704,48.775764,27.158847
1,RR_CH2-dioxane.xyz,18,44,"44,45,46,47,48,49,50,51,52,53,54,55,56",1.528873,3.366524,9.520385,5.212852,47.385756,26.384877
2,RR_CH2CN.xyz,18,44,"44,45,46,47,48",1.539665,3.384188,9.501333,3.800078,43.044206,23.967457
3,RR_CH2CO2Me.xyz,18,44,"44,45,46,47,48,49,50,51,52,53",1.529564,3.356732,9.428808,3.904486,47.555193,26.479221
4,RR_CMe2CO2Me.xyz,18,44,"44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59",1.552641,3.700615,9.432251,4.984558,66.466195,37.009062
5,RR_Cy.xyz,18,44,"44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,5...",1.539015,3.712287,9.604943,5.186902,52.541020,29.255381
6,RR_Et.xyz,18,44,"44,45,46,47,48,49,50",1.529597,3.396047,9.586415,3.929877,42.899871,23.887090
7,RR_Me-Propane.xyz,18,44,"44,45,46,47,48,49,50,51,52,53,54,55,56",1.537140,3.493937,9.577250,4.392814,48.266408,26.875233
8,RR_Me.xyz,18,44,"44,45,46,47",1.525097,3.387621,9.482057,3.861140,32.990971,18.369712
9,RR_Ph.xyz,18,44,"44,45,46,47,48,49,50,51,52,53,54",1.519774,3.639588,9.414987,5.158933,49.854091,27.759271



Excel file saved:
/content/drive/MyDrive/xyz_files/R3_Sterimol_Vbur_results.xlsx
